In [10]:
%tb
import os, re, json, math, time, random, argparse, glob
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, AutoTokenizer
from torch.optim import AdamW

import matplotlib
matplotlib.use("Agg")           # headless — saves PNG files
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import warnings
# warnings.filterwarnings("ignore")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

KeyboardInterrupt: 

[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Tokenizer

In [11]:
# character-level BPE-lite

SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}

class CodeTokenizer:
    """
    Simple sub-word tokenizer tailored for Python source code.
    Splits on whitespace/punctuation, keeps indentation tokens,
    and falls back to characters for unknowns.
    """
    PUNCT = set(",;&|~^@#")

    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.token2id: Dict[str, int] = dict(SPECIAL)
        self.id2token: Dict[int, str] = {v: k for k, v in SPECIAL.items()}
        self.built = False

    # ── build ──────────────────────────────────────────────
    def build(self, texts: List[str], min_freq: int = 3):
        freq: Dict[str, int] = defaultdict(int)
        for t in texts:
            for tok in self._raw_split(t):
                freq[tok] += 1
        sorted_tokens = sorted(freq.items(), key=lambda x: -x[1])
        for tok, cnt in sorted_tokens:
            if cnt < min_freq:
                break
            if tok not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        # fill remaining slots with single chars
        for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,_ \t\n":  #for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_ \t\n":
            if c not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[c] = idx
                self.id2token[idx] = c
        self.built = True
        print(f"[Tokenizer] vocab_size={len(self.token2id)}")

    def _raw_split(self, text: str) -> List[str]:
        tokens = []
        for line in text.splitlines(keepends=True):
            # capture leading whitespace as indent token
            stripped = line.lstrip(" \t")
            indent = line[: len(line) - len(stripped)]
            for ch in indent:
                tokens.append(ch)
            # split remainder on punctuation / spaces
            buf = ""
            for ch in stripped:
                if ch in self.PUNCT or ch in " \t\n\r":
                    if ch == "\n" or ch.strip():
                        if buf:
                            tokens.append(buf)
                        tokens.append(ch)
                        continue
                    if buf:
                        buf += ch
                        tokens.append(buf)
                        buf = ""
                        tokens.append("\n")
                else:
                    buf += ch
            if buf:
                tokens.append(buf)
        return tokens

    def encode(self, text: str) -> List[int]:
        ids = [SPECIAL["<BOS>"]]
        for tok in self._raw_split(text):
            if tok in self.token2id:
                ids.append(self.token2id[tok])
            else:
                # char fallback
                for ch in tok:
                    ids.append(self.token2id.get(ch, SPECIAL["<UNK>"]))
        ids.append(SPECIAL["<EOS>"])
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok in SPECIAL:
                continue
            parts.append(tok)
        return "".join(parts)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump({"token2id": self.token2id}, f)

    @classmethod
    def load(cls, path: str) -> "CodeTokenizer":
        with open(path) as f:
            d = json.load(f)
        obj = cls()
        obj.token2id = {k: int(v) for k, v in d["token2id"].items()}
        obj.id2token = {v: k for k, v in obj.token2id.items()}
        obj.built = True
        return obj

    @property
    def pad_id(self):  return SPECIAL["<PAD>"]
    @property
    def eos_id(self):  return SPECIAL["<EOS>"]
    @property
    def bos_id(self):  return SPECIAL["<BOS>"]
    @property
    def vocab(self):   return len(self.token2id)

## Datasets

In [12]:

def load_files(data_dir: str, max_files: int = 0) -> List[str]:
    """Load .py / .txt files from a directory tree."""
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files:
        files = files[:max_files]
    texts = []
    for fp in files:
        try:
            texts.append(Path(fp).read_text(errors="replace"))
        except Exception:
            pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts



class TokenDataset(Dataset):
    """
    Sliding-window dataset for next-token prediction.
    Target at each position is the next token id.
    """
    def __init__(self, ids: List[int], ctx: int = 128):
        self.ctx = ctx
        self.data = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return max(0, len(self.data) - self.ctx - 1)

    def __getitem__(self, i):
        x = self.data[i: i + self.ctx]
        y = self.data[i + 1: i + self.ctx + 1]
        return x, y


class T5LineDataset(Dataset):
    """
    Each sample: tokenise the prefix with AutoTokenizer → input_ids
                 tokenise the suffix                        → labels
    Padding and label-masking are handled in the collator below.
    """
    def __init__(self, texts: List[str], hf_tokenizer: AutoTokenizer,
                 max_prefix: int = 96, max_suffix: int = 64):
        self.tok = hf_tokenizer
        self.max_prefix = max_prefix
        self.max_suffix = max_suffix
        self.samples: List[Tuple[str, str]] = []

        for text in texts:
            for line in text.splitlines():
                line = line.rstrip()
                if len(line.strip()) < 10:
                    continue
                # split at 30–70 % of the line (your "root cause 2" fix)
                words = line.split()
                if len(words) < 3:
                    continue
                cut = random.randint(
                    max(1, int(len(words) * 0.3)),
                    max(2, int(len(words) * 0.7)),
                )
                prefix = " ".join(words[:cut])
                suffix = " ".join(words[cut:])
                self.samples.append((prefix, suffix))

        print(f"[T5LineDataset] {len(self.samples)} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        prefix, suffix = self.samples[i]
        enc = self.tok(
            prefix,
            max_length=self.max_prefix,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        dec = self.tok(
            suffix,
            max_length=self.max_suffix,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            dec["input_ids"].squeeze(0),
        )



def collate_t5(batch, pad_id: int):
    """Pad input_ids, attention_mask, and labels in a single pass."""
    src_ids, src_masks, lbl_ids = zip(*batch)

    max_src = max(t.size(0) for t in src_ids)
    max_lbl = max(t.size(0) for t in lbl_ids)

    B = len(batch)
    SRC  = torch.full((B, max_src), pad_id, dtype=torch.long)
    MASK = torch.zeros((B, max_src), dtype=torch.long)
    LBL  = torch.full((B, max_lbl), -100,   dtype=torch.long)   # -100 = ignored by T5 loss

    for i, (s, m, l) in enumerate(zip(src_ids, src_masks, lbl_ids)):
        SRC[i,  :s.size(0)] = s
        MASK[i, :m.size(0)] = m
        LBL[i,  :l.size(0)] = l

    return SRC, MASK, LBL

## Models

In [13]:
@dataclass
class ModelCfg:
    vocab: int = 8000
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    d_ff: int = 1024
    max_len: int = 256
    dropout: float = 0.1


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class TokenModel(nn.Module):
    """
    Decoder-only Transformer for causal next-token prediction.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.emb   = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.pos   = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        layer      = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout,
            batch_first=True, norm_first=True
        )
        self.enc   = nn.TransformerEncoder(layer, cfg.n_layers)
        self.head  = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.emb.weight = self.head.weight  # weight tying

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        h = self.pos(self.emb(x))
        h = self.enc(h, mask=mask, is_causal=True)
        return self.head(h)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int,
                 temperature: float = 0.8, top_k: int = 50,
                 stop_at_word_end: bool = True,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev   = next(self.parameters()).device
        ids   = list(prefix_ids)
        generated = []
        PUNCT_CHARS = set("()[]{}.,;:=+-*/\\%<>!&|~^@# \t\n\"'`")
        for _ in range(max_new):
            x = torch.tensor([ids[-self.cfg.max_len:]], dtype=torch.long, device=dev)
            logits = self(x)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            ids.append(nxt)
            generated.append(nxt)
            if stop_at_word_end and tokenizer:
                tok = tokenizer.id2token.get(nxt, "")
                if any(c in PUNCT_CHARS for c in tok):
                    break
        return generated


# class LineModel(nn.Module):
#     """
#     Encoder-Decoder Transformer for seq2seq line completion.
#     Encoder: reads prefix.  Decoder: generates rest of line.
#     """
#     def __init__(self, cfg: ModelCfg):
#         super().__init__()
#         self.cfg = cfg
#         self.enc_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
#         self.dec_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
#         self.enc_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
#         self.dec_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
#         self.transformer = nn.Transformer(
#             cfg.d_model, cfg.n_heads, cfg.n_layers, cfg.n_layers,
#             cfg.d_ff, cfg.dropout, batch_first=True, norm_first=True
#         )
#         self.head = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
#         self.dec_emb.weight = self.head.weight

#     def forward(self, src: torch.Tensor, tgt: torch.Tensor,
#                 src_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
#         T = tgt.size(1)
#         causal = nn.Transformer.generate_square_subsequent_mask(T, device=src.device)
#         enc_out = self.transformer.encoder(
#             self.enc_pos(self.enc_emb(src)),
#             src_key_padding_mask=src_key_padding_mask
#         )
#         dec_out = self.transformer.decoder(
#             self.dec_pos(self.dec_emb(tgt)),
#             enc_out,
#             tgt_mask=causal,
#             tgt_is_causal=True,
#             memory_key_padding_mask=src_key_padding_mask
#         )
#         return self.head(dec_out)

#     @torch.no_grad()
#     def generate(self, prefix_ids: List[int], max_new: int = 64,
#                  temperature: float = 0.7, top_k: int = 40,
#                  tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
#         self.eval()
#         dev = next(self.parameters()).device
#         src = torch.tensor([prefix_ids], dtype=torch.long, device=dev)
#         dec_ids = [SPECIAL["<BOS>"]]
#         out_ids = []
#         for _ in range(max_new):
#             tgt = torch.tensor([dec_ids], dtype=torch.long, device=dev)
#             logits = self(src, tgt)[0, -1] / temperature
#             if top_k:
#                 topk_v, _ = torch.topk(logits, top_k)
#                 logits[logits < topk_v[-1]] = -float("inf")
#             probs = F.softmax(logits, dim=-1)
#             nxt = torch.multinomial(probs, 1).item()
#             if nxt == SPECIAL["<EOS>"]:
#                 break
#             dec_ids.append(nxt)
#             out_ids.append(nxt)
#         return out_ids

## Metrics, Visualisation

In [14]:
@dataclass
class MetricLog:
    train_loss:  List[float] = field(default_factory=list)
    val_loss:    List[float] = field(default_factory=list)
    train_ppl:   List[float] = field(default_factory=list)
    val_ppl:     List[float] = field(default_factory=list)
    lr:          List[float] = field(default_factory=list)
    token_acc:   List[float] = field(default_factory=list)   # top-1 accuracy
    grad_norm:   List[float] = field(default_factory=list)

    def append(self, **kw):
        for k, v in kw.items():
            getattr(self, k).append(v)


def plot_metrics(log: MetricLog, title: str, save_path: str):
    """
    Rich 2×3 dashboard saved to PNG.
    """
    epochs = list(range(1, len(log.train_loss) + 1))
    fig = plt.figure(figsize=(18, 10), facecolor="#0d1117")
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    ACCENT = "#58a6ff"
    WARN   = "#f78166"
    OK     = "#3fb950"
    GRID   = "#21262d"
    TXT    = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor":   "#161b22",
        "axes.edgecolor":   GRID,
        "axes.labelcolor":  TXT,
        "xtick.color":      TXT,
        "ytick.color":      TXT,
        "text.color":       TXT,
        "grid.color":       GRID,
        "grid.linewidth":   0.6,
    })

    def _ax(pos, ylabel, title_s):
        ax = fig.add_subplot(pos)
        ax.set_xlabel("Epoch", fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(title_s, fontsize=10, color=ACCENT, pad=6)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(True)
        return ax

    # 1 — Loss
    ax = _ax(gs[0, 0], "Loss", "Train / Val Loss")
    ax.plot(epochs, log.train_loss, color=ACCENT, lw=1.8, label="train")
    if log.val_loss:
        ax.plot(epochs, log.val_loss, color=WARN, lw=1.8, linestyle="--", label="val")
    ax.legend(fontsize=8)

    # 2 — Perplexity
    ax = _ax(gs[0, 1], "Perplexity", "Train / Val Perplexity")
    ax.plot(epochs, log.train_ppl, color=ACCENT, lw=1.8, label="train")
    if log.val_ppl:
        ax.plot(epochs, log.val_ppl, color=WARN, lw=1.8, linestyle="--", label="val")
    ax.set_yscale("log")
    ax.legend(fontsize=8)

    # 3 — Token top-1 accuracy
    ax = _ax(gs[0, 2], "Accuracy", "Top-1 Token Accuracy")
    ax.plot(epochs, log.token_acc, color=OK, lw=1.8)
    ax.set_ylim(0, 1)

    # 4 — LR schedule
    ax = _ax(gs[1, 0], "LR", "Learning Rate")
    ax.plot(epochs, log.lr, color="#d2a8ff", lw=1.5)
    ax.set_yscale("log")

    # 5 — Gradient norm
    ax = _ax(gs[1, 1], "Grad Norm", "Gradient Norm")
    ax.plot(epochs, log.grad_norm, color="#ffa657", lw=1.5)

    # 6 — Train vs Val gap (over-fit indicator)
    ax = _ax(gs[1, 2], "Δ Loss (train-val)", "Generalisation Gap")
    if log.val_loss:
        gap = [v - t for t, v in zip(log.train_loss, log.val_loss)]
        ax.fill_between(epochs, 0, gap,
                        where=[g > 0 for g in gap], color=WARN, alpha=0.35, label="overfit")
        ax.fill_between(epochs, 0, gap,
                        where=[g <= 0 for g in gap], color=OK, alpha=0.35, label="underfit")
        ax.plot(epochs, gap, color=TXT, lw=1.0)
        ax.axhline(0, color=GRID, lw=1)
        ax.legend(fontsize=8)

    fig.suptitle(title, fontsize=14, color=ACCENT, y=1.01)
    plt.savefig(save_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Plot] saved → {save_path}")

## Checkpoints

In [15]:
class BestModelSaver:
    """Keeps the best N checkpoints by val loss."""
    def __init__(self, ckpt_dir: str, model_name: str, keep: int = 3):
        self.dir   = Path(ckpt_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.name  = model_name        
        self.keep  = keep
        self.saved: List[Tuple[float, str]] = []  # (val_loss, path)

    def save(self, model: nn.Module, val_loss: float, epoch: int, extra: dict = None):
        path = str(self.dir / f"{self.name}_ep{epoch:03d}_loss{val_loss:.4f}.pt")
        payload = {
            "model_state": model.state_dict(),
            "val_loss":    val_loss,
            "epoch":       epoch,
            "cfg":         getattr(model, "cfg", None),
        }
        if extra:
            payload.update(extra)
        torch.save(payload, path)
        self.saved.append((val_loss, path))
        self.saved.sort(key=lambda x: x[0])
        while len(self.saved) > self.keep:
            _, old = self.saved.pop()
            try:   os.remove(old)
            except FileNotFoundError: pass
            print(f"[Saver] removed old ckpt: {old}")
        print(f"[Saver] saved ckpt: {path}  (val_loss={val_loss:.4f})")

    def best_path(self) -> Optional[str]:
        return self.saved[0][1] if self.saved else None

## Training loop

In [27]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_token_model(
    model:      TokenModel,
    train_dl:   DataLoader,
    val_dl:     DataLoader,
    epochs:     int,
    lr:         float,
    device:     torch.device,
    saver:      BestModelSaver,
    log:        MetricLog,
    plot_dir:   str,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])

    for ep in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        for x, y in tqdm(train_dl, desc=f"[Token] Epoch {ep}/{epochs} train",
                 leave=False, unit="batch"):
            # if index % one_part == 0:
                # print(f"{index // one_part}% at {datetime.now().time()}")
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = crit(logits.view(-1, logits.size(-1)), y.view(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss += loss.item()
            preds = logits.argmax(-1)
            mask = (y != SPECIAL["<PAD>"])
            t_acc += (preds[mask] == y[mask]).float().mean().item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc / t_steps

        # ── val ──────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for x, y in tqdm(val_dl, desc=f"[Token] Epoch {ep}/{epochs} val  ",
                 leave=False, unit="batch"):
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        print(f"[Token ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
              f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}  lr={opt.param_groups[0]['lr']:.2e}")

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"Token Model — Epoch {ep}", f"{plot_dir}/token_metrics_ep{ep:03d}.png")

    plot_metrics(log, "Token Model — Final", f"{plot_dir}/token_metrics_final.png")



def train_line_model(
    model:    T5ForConditionalGeneration, 
    hf_tok:   AutoTokenizer,             # HuggingFace tokenizer
    train_dl: DataLoader,
    val_dl:   DataLoader,
    epochs:   int,
    lr:       float,
    device:   torch.device,
    saver:    BestModelSaver,
    log:      MetricLog,
    plot_dir: str,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    # T5 has its own internal cross-entropy; we just call model(...).loss
    opt   = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)

    for ep in range(1, epochs + 1):
        # ── TRAIN ────────────────────────────────────────────────────────────
        model.train()
        t_loss = t_acc = t_steps = 0
        gn = 0.0

        batch_bar = tqdm(train_dl,
                         desc=f"[Line] Epoch {ep}/{epochs} train",
                         leave=False, unit="batch")

        for src, mask, lbl in batch_bar:
            src, mask, lbl = src.to(device), mask.to(device), lbl.to(device)

            out  = model(input_ids=src, attention_mask=mask, labels=lbl)
            loss = out.loss          # T5 computes CE internally, -100 labels ignored

            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()

            # token accuracy: compare argmax logits vs labels where label != -100
            with torch.no_grad():
                preds = out.logits.argmax(-1)
                valid = (lbl != -100)
                acc   = (preds[valid] == lbl[valid]).float().mean().item()

            t_loss  += loss.item()
            t_acc   += acc
            t_steps += 1

            batch_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.3f}")

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        # ── VAL ──────────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0

        with torch.no_grad():
            for src, mask, lbl in tqdm(val_dl,
                                       desc=f"[Line] Epoch {ep}/{epochs} val  ",
                                       leave=False, unit="batch"):
                src, mask, lbl = src.to(device), mask.to(device), lbl.to(device)
                out    = model(input_ids=src, attention_mask=mask, labels=lbl)
                v_loss  += out.loss.item()
                v_steps += 1

        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        # ── LOGGING ──────────────────────────────────────────────────────────
        log.append(
            train_loss=tl, val_loss=vl,
            train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
            lr=opt.param_groups[0]["lr"],
            token_acc=ta, grad_norm=gn,
        )
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        # save best checkpoint — store HF model with torch.save so BestModelSaver
        # stays unchanged; pass the raw state dict the same way
        saver.save(model, vl, ep)

        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"Line Model — Epoch {ep}",
                         f"{plot_dir}/line_metrics_ep{ep:03d}.png")

    plot_metrics(log, "Line Model — Final", f"{plot_dir}/line_metrics_final.png")

## Hand Testing

In [28]:
def hand_test_repl(token_model: TokenModel, line_model: T5ForConditionalGeneration,
                   tokenizer: CodeTokenizer, hf_tok: AutoTokenizer,
                   device: torch.device):
    print("  Python Autocomplete — Interactive Test")
    print("  Commands: :token <prefix>  |  :line <prefix>")
    print("            :temp <float>   |  :k <int>  |  :quit")

    temperature = 0.3    # lower default — better for code
    top_k       = 10

    while True:
        try:
            raw = input(">> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye!")
            break

        if not raw:
            continue

        if raw.startswith(":quit"):
            break
        elif raw.startswith(":temp"):
            try:   temperature = float(raw.split()[1])
            except: print("Usage: :temp 0.7")
            print(f"temperature = {temperature}")
            continue
        elif raw.startswith(":k"):
            try:   top_k = int(raw.split()[1])
            except: print("Usage: :k 40")
            print(f"top_k = {top_k}")
            continue
        elif raw.startswith(":token"):
            prefix = raw[6:].strip()
            ids = tokenizer.encode(prefix)[:-1]
            with torch.no_grad():
                new_ids = token_model.generate(
                    ids, max_new=20, temperature=temperature,
                    top_k=top_k, stop_at_word_end=True, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← token completion: {prefix}\033[32m{completion}\033[0m\n")
        else:
            prefix = raw[5:].strip() if raw.startswith(":line") else raw
            inp = hf_tok(prefix, return_tensors="pt").to(device)
            with torch.no_grad():
                out = line_model.generate(
                    **inp,
                    max_new_tokens=64,
                    temperature=temperature,
                    do_sample=(temperature > 0.1),
                    top_k=top_k,
                )
            completion = hf_tok.decode(out[0], skip_special_tokens=True)
            print(f"  ← line  completion: {prefix}\033[33m{completion}\033[0m\n")

## Main

In [ ]:
class Arguments():
    def __init__(self, data_dir: str = "Clean_Dataset", ckpt_dir: str = "checkpoints",
                    plot_dir: str = "plots", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    # args = Arguments()
    # args = Arguments(max_files=100)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    torch.serialization.add_safe_globals([ModelCfg])

    HF_MODEL = "Salesforce/codet5-small"
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        tm = TokenModel(cfg).to(device)
        tok_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / "token_model_*.pt")))
        if tok_paths:
            ck = torch.load(tok_paths[0], map_location=device, weights_only=False)
            tm.load_state_dict(ck["model_state"])
            print(f"[Loaded] token model from {tok_paths[0]}")
    
        lm = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / "line_model_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
    
        hand_test_repl(tm, lm, tokenizer, hf_tok, device)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]
    
    # ── TOKEN MODEL ──────────────────────────────────────────
    if not args.skip_token:
        print("  Prepairing TOKEN model")

        # flatten all train text → single id stream
        all_ids_tr = []
        for t in tr_txt:
            all_ids_tr.extend(tokenizer.encode(t))
        all_ids_va = []
        for t in va_txt:
            all_ids_va.extend(tokenizer.encode(t))

        tr_ds = TokenDataset(all_ids_tr, args.ctx)
        va_ds = TokenDataset(all_ids_va, args.ctx)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,  num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False, num_workers=0, pin_memory=True)

        tok_model = TokenModel(cfg).to(device)
        n_params  = sum(p.numel() for p in tok_model.parameters() if p.requires_grad)
        print(f"[Token Model] {n_params/1e6:.2f}M parameters")

        tok_saver = BestModelSaver(args.ckpt_dir, "token_model")
        tok_log   = MetricLog()
        print("  Training TOKEN model")
        train_token_model(tok_model, tr_dl, va_dl, args.epochs, args.lr,
                          device, tok_saver, tok_log, args.plot_dir)
    else:
        tok_model = TokenModel(cfg).to(device)
        tok_saver = BestModelSaver(args.ckpt_dir, "token_model")
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / "token_model_*.pt")))
        if paths:
            ck = torch.load(paths[0], map_location=device)
            tok_model.load_state_dict(ck["model_state"])

    # ── LINE MODEL ───────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        line_model = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        
        collate    = lambda b: collate_t5(b, hf_tok.pad_token_id)
        tr_line_ds = T5LineDataset(tr_txt, hf_tok)
        va_line_ds = T5LineDataset(va_txt, hf_tok)
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.1f}M parameters (codet5-small)")
        
        line_saver = BestModelSaver(args.ckpt_dir, "line_model")
        line_log   = MetricLog()
        train_line_model(line_model, hf_tok, tr_line_dl, va_line_dl,
                         args.epochs, args.lr, device, line_saver, line_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(tok_model, line_model, tokenizer, hf_tok, device)


main()

[Tokenizer] loading tokenizer.json
[Loading] Started loading
[Data] loaded 12110 files from Clean_Dataset
[Loading] Ended loading
  Prepairing LINE model
[T5LineDataset] 676196 samples
[T5LineDataset] 73886 samples
[Line  Model] 60.5M parameters (codet5-small)
[Line] DataLoader — 21132 train batches, 2309 val batches


[Line  ep   1] train_loss=2.3719  val_loss=2.1270  ppl=8.4  acc=0.566  lr=4.55e-04
[Saver] saved ckpt: checkpoints\line_model_ep001_loss2.1270.pt  (val_loss=2.1270)
[Plot] saved → plots/line_metrics_ep001.png


[Line  ep   2] train_loss=2.0100  val_loss=2.0267  ppl=7.6  acc=0.614  lr=3.36e-04
[Saver] saved ckpt: checkpoints\line_model_ep002_loss2.0267.pt  (val_loss=2.0267)
[Plot] saved → plots/line_metrics_ep002.png


[Line  ep   3] train_loss=1.8088  val_loss=1.9565  ppl=7.1  acc=0.643  lr=1.89e-04
[Saver] saved ckpt: checkpoints\line_model_ep003_loss1.9565.pt  (val_loss=1.9565)
[Plot] saved → plots/line_metrics_ep003.png


[Line  ep   4] train_loss=1.6322  val_loss=1.9069  ppl=6.7  acc=0.670  lr=7.04e-05
[Saver] removed old ckpt: checkpoints\line_model_ep001_loss2.1270.pt
[Saver] saved ckpt: checkpoints\line_model_ep004_loss1.9069.pt  (val_loss=1.9069)
[Plot] saved → plots/line_metrics_ep004.png


...

  ← line  completion: with open(# noqa: E402



>>  def fo


  ← line  completion: def fo= fo.get("foobar"):



>>  def foo(


  ← line  completion: def foo(# noqa: E402

